## Docker lab assignment

Welcome to this lab where you'll apply your Docker skills. Remember, Docker is a powerful tool for creating, deploying, and running applications in containers. These containers encapsulate an application along with its dependencies, ensuring consistent runs across different environments.

This repository includes both backend and frontend applications. Our backend, similar to the FastAPI app developed in previous labs, uses the Stable Diffusion API to create images from user inputs. The frontend is a straightforward Streamlit app for interacting with the backend.


**Our primary objective is to construct a Docker Compose file that runs both applications in a containerized setup.**


First, we'll examine our FastAPI backend and conduct some tests. Then, we'll explore the Streamlit frontend, observing how the two applications interact. After testing both applications, we'll begin the process of dockerizing them individually before creating a Docker Compose file for their joint operation.

Let's get started!

## Backend App
The backend is a basic FastAPI app using Stable Diffusion API to generate images from user prompts. Have a look at code in the backend dir - start by installing the necessary dependencies listed in requirements.txt:

```bash
pip install -r requirements.txt
```

Have a look at the source code, specifically the code in routes/image_gen_router.

There are four different endpoints:

`POST /images/` - Generates an image based on a user prompt. The user prompt should be passed in the body of the request. Synchronous request responds with the image.

`POST /v2/image/` - Generates an image based on a user prompt. The user prompt should be passed in the body of the request. Asynchronous request using the fastAPI background_tasks api and responds with the image id.

`POST /v3/image/` - Generates an image based on a user prompt. The user prompt should be passed in the body of the request. Asynchronous request using a redis queue api and responds with the image id.

`GET /image/{image_id}` - Returns the image based on the image id. If the image is not yet generated it will return a 202.


Run the app inside the backend dir using uvicorn:

```bash
uvicorn app:app --reload
```

If you've postman installed you can now test all different endpoints using the following body:
    
```json
{
    "userPrompt": "<add your prompt here>"
}
```
   
If you don't have postman installed you can use curl to test the endpoints eg:

```bash
curl -X POST "http://localhost:8000/images/" -H  "accept: application/json" -H  "Content-Type: application/json" -d "{\"userPrompt\":\"<add your prompt here>\"}"
```


## Frontend App / Streamlit App

After you've tested the backend app, you can now checkout the frontend app. The frontend app is a simple streamlit app that allows you to interact with the backend app. The frontend app is located in the frontend dir.
Have a look at the source code, specifically the code in streamlit_app.py. Try to understand how the comminucation between the frontend and backend app works. You'll see that there'se an environment variable called API_URL. This variable is used to set the url of the backend app. Have a look at the different implementations and interactions with the different endpoints.

Make sure you install all requirements in the requirements.txt file.

You can now run the streamlit app using the following command:


```bash
streamlit run streamlit_app.py
```

The streamlit app should now be running on http://localhost:8501. You can now interact with the app. The app allows you to generate images based on a user prompt synchronously and asynchronously. 

Give it a few test runs


## Dockerizing the backend app

Your first task now is to dockerize the backend app. So, go to the backend dir, create an adequate Dockerfile, build it and run the container. 

Have a look at the Dockerfile reference from previous classes to see how to create a Dockerfile.

Ask yourself the following questions:

- What base image should I use?
- What files do I need to copy to the container?
- What commands do I need to run to install the requirements?
- What command do I need to run to start the app?
- What port do I need to expose?
- What command do I need to run to start the app?

After you've created the Dockerfile you can build the image using the following command:

A good image name could be `backend-img-generator`.

```bash
docker build -t <your image name> .
```

After the image is built you can run the image using the following command:

```bash
docker run -p 8000:8000 <your image name>
```

You can now test the app using curl or postman.

## Dockerizing the frontend app

Before dockerizing the frontend app, give it a few test runs, try letting it interact with the backend app running in a containerized environment.


Proceed with dockerizing the frontend app similar to the backend app. Create dockerfile inside the frontend dir and build the image.

Build the image using the following command:

A good image name could be `frontend-img-generator`.

```bash
docker build -t <your image name> .
```

After the image is built you can run the image using the following command:


```bash
docker run -p 8501:8501 <your image name>
```

Now if you want to both, the image generator and the frontend app running in a containerized environment, you'll have to run two containers.

You can either start both containers manually, or you can use docker-compose to start both containers.

If you want to start both containers manually, you'll have to start the image generator container first, and then the frontend container. You'll have to make sure that the API_URL environment variable is set correctly in the frontend container.

```bash
docker run -p 8000:8000 <your image generator image name>
docker run -p 8501:8501 <your frontend image name>
```

Are you able to get both containers running? Is the streamlit able to communicate with the backend app? You may notice that the streamlit app is not able to communicate with the backend app. Do you know why? 



Pause for a second and think what could be the reason?



Remember, how we've been talking about container isolation? Each container has it's own network stack. So, the streamlit app is not able to communicate with the backend app because they are running in different containers and therefore different network stacks.

We can connect the two containers using docker networking. We can create a docker network and connect both containers to the same network. This way both containers will be able to communicate with each other.

### Create your custom network

If you have both containers still running, it maybe a good idea, to stop and remove them all now.

Use following commands to stop and remove all running containers:

- **Stop all containers:** `docker stop $(docker ps -q)`
- **Remove all containers:** `docker rm $(docker ps -aq)`

#### Consider the API_URL before creating a docker network

You'll have to make sure that the API_URL environment variable is set correctly in the frontend container.


**You have different options for that**

- Set the environment variable directly in the dockerfile of the frontend using following dockerfile instruction: 
`ENV API_URL=http://fast:8000`
Remember to rebuild the frontend image


- Edit the .env file and run the container with
`--env-file=.env` inside the `frontend` dir

- Run the frontend container with `-e`flag:
`docker run -e API_URL=`

- Update the .env file and rebuild the **image** with the edited .env variable.


## Create a docker network and connect containers

You can create a docker network using the following command:

```bash
docker network create <your network name>
```


### Connect multiple docker containers to same network

You have 2 options to connect containers to a network.

- **Option 1**: 
Create and attach a container to network at the same time. Simply run a container and connect if afterwards

- **Option 2**: 
First run all containers using the corresponding images. - And connect them after those containers where created


**Important**
Depending on how you're going to decide to proceed with the `API_URL` param, you may have to run the frontend container with additional params

### Option 1 Run and attach at the same time
Run start both containers on the same network using the following command:

Remember you can always use `-d` to run them in background!

```bash
docker run -p 8000:8000 --network <your network name> <image>
docker run -p 8501:8501 --network <your network name> <image>
```

##### Option 2 Connect already running containers

**First ensure your containers are running.**

You can connect a container to a network using the following command:

```bash
docker network connect <your network name> <your container name>
```

You can check if the containers are connected to the network using the following command:

```bash
docker network inspect <your network name>
```

Seems quite a lot of work to get both containers running. Wouldn't it be nice if we could start both containers with a single command?
 
This is where docker-compose comes in and makes our lives easier.


## Docker Compose Integration

Now that you have successfully dockerized both the backend and frontend applications, the next task is to create and test a Docker Compose file. Docker Compose is a tool for defining and running multi-container Docker applications. With a single command, you can spin up both the frontend and backend services in a connected and consistent environment.

### Objective

Your task is to write a `docker-compose.yml` file that defines the configurations for both the backend and frontend services, ensuring they can communicate with each other effectively.

### Steps

1. **Create the Docker Compose File**: In the root directory of your project, create a file named `docker-compose.yml`.

2. **Define Services**:
   - Define two services: `backend` and `frontend`.
   - For each service, specify the Docker image to use (either using the images you built earlier or building them directly within the Docker Compose file).
   - Ensure the `frontend` service knows how to reach the `backend` service, typically through environment variables or Docker Compose's networking features.

3. **Set Up Networking**:
   - Configure Docker Compose networking to enable communication between the two services. By default, Docker Compose sets up a single network for all services in the `docker-compose.yml` file, and each service is reachable by other services at a hostname identical to the service name.

4. **Expose Ports**:
   - Expose the necessary ports for each service. For the backend, this might be port 8000, and for the frontend, it could be port 8501.

5. **Environment Configuration**:
   - Include any environment variables required by either service. For instance, the frontend might need an environment variable to locate the backend service.

6. **Volumes**:
   - If your applications require persistent storage or you want to use Docker volumes for live code reloading during development, define the necessary volumes.

### Testing the Docker Compose Setup

After writing your `docker-compose.yml` file, test it to ensure both services run correctly:

1. **Build and Run**:
   - Use `docker-compose up --build` to build and start the services defined in your Docker Compose file.

2. **Verify Functionality**:
   - Check if the frontend is accessible at `http://localhost:8501` and is successfully communicating with the backend.
   - Test the backend independently to confirm it's functioning as expected.

3. **Debugging**:
   - If you encounter issues, use `docker-compose logs` to check the logs of your services for debugging.
   - Make sure that the services are correctly configured and networked.

